# 01 - Exploratory Data Analysis - Data Cleaning

this notebook aims to clean the data supplied to us for the analysis and writes it to a pickle file, enabling faster loading of the data provided compared to loading from the original source xlsx files. 


In [1]:
# this file is used to find the project root and set the working directory to it.
from pathlib import Path

def find_project_root(marker=".project-root"):
    path = Path.cwd().resolve()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(f"Could not find {marker}")

PROJECT_ROOT = find_project_root()
print(f"Project root found at: {PROJECT_ROOT}")

cwd = PROJECT_ROOT

import pandas as pd 

Project root found at: /Users/ducjeremyvu/mime/sem-1/files/bank-fintech/essay


## Data loading and saving into a more efficient format
we have 2 xlxs files, one containing the full dataset and the other containing further information on the columns. 
the first cell loads the xlsx files initially into a pandas dataframe and then saves them as pickle files for faster loading later on.
the second cell with then be used for loading the pickle files after successfully writing the data into pickle format. 

In [2]:
path_data = 'data/babina_et_al_merged_data.xlsx'
path_descr = 'data/Babina_et_al_var_description_KR.xlsx'

# might require to install openpyxl package to read excel files

data = pd.read_excel(cwd / path_data)
descr = pd.read_excel(cwd / path_descr)

# saving to pickle format for faster loading in the future 
data.to_pickle(cwd / "data/babina.pkl")
descr.to_pickle(cwd / "data/description.pkl")

In [3]:
# reading the pickle format 

# even though it is not necessar, we save the original raw dataset 
# into a pickle file, in case we wish to alter changes in the cleaning step. 

data = pd.read_pickle(cwd / "data/babina.pkl")
descr = pd.read_pickle(cwd / "data/description.pkl")

In [18]:
# brief check of the data and description files
descr.head(10)

,Variable,Variable description
0,gvkey,Firm unique identifier
1,year,Year
2,aiempl,(mean) total_ai_employment (Cognism)
3,totalempl,(mean) total_employment (Cognism)
4,act,Current Assets - Total
5,ap,Accounts Payable - Trade
6,apalch,Accounts Payable and Accrued Liabilities - Inc...
7,apb,"Accounts Payable/Creditors - Brokers, Dealers,..."
8,apc,Accounts Payable/Creditors - Customer
9,apofs,Accounts Payable/Creditors - Other - FS


In [17]:

pd.options.display.max_columns = 200
data.describe().T[['mean', 'std', 'max' ]].iloc[:10]

,mean,std,max
gvkey,76430.701948,73493.815943,3.287950e+05
year,2011.316740,3.975979,2.018000e+03
aiempl,8.094386,97.511928,6.190000e+03
totalempl,6576.617004,27176.778743,1.552543e+06
act,5424.100555,3114.470811,1.080283e+04
ap,6240.781483,3612.951506,1.249393e+04
apalch,113.711461,165.823404,4.019948e+02
apb,NaN,NaN,NaN
apc,NaN,NaN,NaN
apofs,NaN,NaN,NaN


In [20]:
data.shape

(50622, 104)

In [6]:
# columns in data summary only count 102 while the original data has 104 columns which is in line with the number of description entries  
len(data.describe().columns) == len(data.columns) 

False

In [7]:
missing_from_describe = data.columns.difference(data.describe().columns)
print(missing_from_describe)

Index(['naics', 'state'], dtype='str')


In [19]:
data[['naics', 'state']].head()

,naics,state
0,322121,XX
1,322121,XX
2,322121,XX
3,322121,XX
4,322121,XX


## Removing Nans 

after quick inspection, we were able to spot columns with nan values, which are basically just empty cells. some of them even have ONLY nan values, meaning that they dont contain any value and can be immediately dropped. this serves in reducing the  

In [9]:
# removing the nans 

nan_columns = data.isna().sum()/len(data)
nan_columns_list = nan_columns[nan_columns == 1].index

In [10]:
# ina ddition to the nan values we include further columns that contain no substantial value (same value or 0 in every row)
additional_drops = ['sic', 'permno', 'state'] 

# adding to the list of columns to drop 
drop_cols = [*nan_columns_list, *additional_drops]
data[drop_cols].describe()

print(f"columns to drop: {drop_cols}")

columns to drop: ['apb', 'apc', 'apofs', 'arb', 'arc', 'artfs', 'dvpd', 'xagt', 'xstf', 'xstfws', 'xt', 'comnam', 'sic', 'permno', 'state']


In [11]:
# Apply the column drops to the dataset
data_cleaned = data.drop(columns=drop_cols)
print(f"Original shape: {data.shape}")
print(f"Cleaned shape: {data_cleaned.shape}")
print(f"Columns dropped: {len(drop_cols)}")

Original shape: (50622, 104)
Cleaned shape: (50622, 89)
Columns dropped: 15


In [12]:
# save the cleaned dataset to a new pickle file for future use
data_cleaned.to_pickle(cwd / "data/babina_cleaned.pkl")

In [13]:
data_cleaned = pd.read_pickle(cwd / "data/babina_cleaned.pkl")